In [ ]:
-- 2. RANK() IT HANDLES THE TIES AND IT HAS THE GAP 

--  Rank the orders based on their sales from highest to lowest

select 
    orderid,
    productid,
    sales,
    Rank() over(order by sales desc) salesrank_rank
from sales.orders

(10 rows affected)

orderid | productid | sales | salesrank_rank
--------+-----------+-------+---------------
8       | 101       | 90    | 1             
4       | 105       | 60    | 2             
10      | 102       | 60    | 2             
6       | 104       | 50    | 4             
7       | 102       | 30    | 5             
5       | 104       | 25    | 6             
9       | 101       | 20    | 7             
3       | 101       | 20    | 7             
2       | 102       | 15    | 9             
1       | 101       | 10    | 10            
(10 rows)

Total execution time: 00:00:01.004

In [ ]:
-- DENSE_RANK() it has the ties and it doesn't leave any gap
select 
    orderid,
    productid,
    sales,
    DENSE_RANK() over(order by sales desc) as denserank_rank
from sales.orders

(10 rows affected)

orderid | productid | sales | denserank_rank
--------+-----------+-------+---------------
8       | 101       | 90    | 1             
4       | 105       | 60    | 2             
10      | 102       | 60    | 2             
6       | 104       | 50    | 3             
7       | 102       | 30    | 4             
5       | 104       | 25    | 5             
9       | 101       | 20    | 6             
3       | 101       | 20    | 6             
2       | 102       | 15    | 7             
1       | 101       | 10    | 8             
(10 rows)

Total execution time: 00:00:00.003

In [ ]:
-- USE CASE 1 : TOP-N ANALYSIS (Analysis the top performers to do targeted marketing)
-- find the top highest sales for each product
SELECT * 
from (      
    select 
        orderid,
        productid,
        sales,
        ROW_NUMBER() OVER (PARTITION BY productid ORDER BY sales DESC) rankbyproduct
    from sales.orders
)t  WHERE rankbyproduct = 1

(4 rows affected)

orderid | productid | sales | rankbyproduct
--------+-----------+-------+--------------
8       | 101       | 90    | 1            
10      | 102       | 60    | 1            
6       | 104       | 50    | 1            
4       | 105       | 60    | 1            
(4 rows)

Total execution time: 00:00:00.031

In [ ]:
-- USE CASE 2 :  BOTTOM-N ANALYSIS (Help analysis the underperformance to manage risks and to do optimizations)
-- find the lowest 2 customers based on their total sales 
SELECT *
FROM(
    SELECT
        CustomerID,
        SUM(Sales) totalsales,
        ROW_NUMBER() OVER (ORDER BY SUM(Sales)) RankCustomers
    from Sales.Orders
    GROUP BY CustomerID
)t WHERE RankCustomers <= 2

(2 rows affected)

CustomerID | totalsales | RankCustomers
-----------+------------+--------------
2          | 55         | 1            
4          | 90         | 2            
(2 rows)

Total execution time: 00:00:00.038

In [ ]:
-- USE CASE 3 : ASSIGN UNIQUE IDS (Help to assign the unique identifier for each how to help paginating)
-- Paginating is the process of breakin down a large data into smaller, more manageable chunks
-- Assign unique IDS to the rows of the 'Orders Archive' 

SELECT
ROW_NUMBER() OVER (ORDER BY orderid, orderdate) uniqueid,
 * 
FROM Sales.OrdersArchive

(10 rows affected)

uniqueid | OrderID | ProductID | CustomerID | SalesPersonID | OrderDate  | ShipDate   | OrderStatus | ShipAddress      | BillAddress    | Quantity | Sales | CreationTime               
---------+---------+-----------+------------+---------------+------------+------------+-------------+------------------+----------------+----------+-------+----------------------------
1        | 1       | 101       | 2          | 3             | 2024-04-01 | 2024-04-05 | Shipped     | 123 Main St      | 456 Billing St | 1        | 10    | 2024-04-01 12:34:56.0000000
2        | 2       | 102       | 3          | 3             | 2024-04-05 | 2024-04-10 | Shipped     | 456 Elm St       | 789 Billing St | 1        | 15    | 2024-04-05 23:22:04.0000000
3        | 3       | 101       | 1          | 4             | 2024-04-10 | 2024-04-25 | Shipped     | 789 Maple St     | 789 Maple St   | 2        | 20    | 2024-04-10 18:24:08.0000000
4        | 4       | 105       | 1          | 3        

In [ ]:
--  USE CASE 4 : ROW_NUMBER() IDENTIFY DUPLICATES (Identify and remove duplicate rows to improve data quality)
-- Q. Identify duplicate rows in the table 'Orders Archive' and return a clean result without any duplicates 
SELECT * 
FROM(
    SELECT 
    ROW_NUMBER() OVER (PARTITION BY orderid ORDER BY creationtime DESC) rn,
    *
    FROM Sales.OrdersArchive
)t where rn = 1

(7 rows affected)

rn | OrderID | ProductID | CustomerID | SalesPersonID | OrderDate  | ShipDate   | OrderStatus | ShipAddress      | BillAddress    | Quantity | Sales | CreationTime               
---+---------+-----------+------------+---------------+------------+------------+-------------+------------------+----------------+----------+-------+----------------------------
1  | 1       | 101       | 2          | 3             | 2024-04-01 | 2024-04-05 | Shipped     | 123 Main St      | 456 Billing St | 1        | 10    | 2024-04-01 12:34:56.0000000
1  | 2       | 102       | 3          | 3             | 2024-04-05 | 2024-04-10 | Shipped     | 456 Elm St       | 789 Billing St | 1        | 15    | 2024-04-05 23:22:04.0000000
1  | 3       | 101       | 1          | 4             | 2024-04-10 | 2024-04-25 | Shipped     | 789 Maple St     | 789 Maple St   | 2        | 20    | 2024-04-10 18:24:08.0000000
1  | 4       | 105       | 1          | 3             | 2024-04-20 | 2024-04-25 | Deli

In [ ]:
-- NTILE(n) OVER (ORDE BY Sales DESC) where the n is number of buckets to be created (formula = rows / bucket_size)
SELECT 
    OrderID,
    Sales,
    NTILE(3) OVER (ORDER BY sales DESC) onebucket
FROM Sales.Orders

(10 rows affected)

OrderID | Sales | onebucket
--------+-------+----------
8       | 90    | 1        
4       | 60    | 1        
10      | 60    | 1        
6       | 50    | 1        
7       | 30    | 2        
5       | 25    | 2        
9       | 20    | 2        
3       | 20    | 3        
2       | 15    | 3        
1       | 10    | 3        
(10 rows)

Total execution time: 00:00:00.009

In [ ]:
-- USE CASE : Date Segmentation : (Divides a dataset into distinct subsets based on the certain criteria) 
-- Segment all orders into 3 categories: high, medium and low sales 
SELECT 
*,
CASE WHEN Buckets = 1 THEN 'High'
     WHEN Buckets = 2 THEN 'Medium'
     WHEN Buckets = 3 THEN 'Low'
END SalesSegmentations
FROM( 
    SELECT 
        OrderID,
        Sales,
        NTILE(3) OVER (ORDER BY  Sales DESC) Buckets
    FROM Sales.Orders
)t 

(10 rows affected)

OrderID | Sales | Buckets | SalesSegmentations
--------+-------+---------+-------------------
8       | 90    | 1       | High              
4       | 60    | 1       | High              
10      | 60    | 1       | High              
6       | 50    | 1       | High              
7       | 30    | 2       | Medium            
5       | 25    | 2       | Medium            
9       | 20    | 2       | Medium            
3       | 20    | 3       | Low               
2       | 15    | 3       | Low               
1       | 10    | 3       | Low               
(10 rows)

Total execution time: 00:00:00.003

In [ ]:
-- USE CASE Equalising Load 
SELECT 
    NTILE(2) OVER (ORDER BY orderid) Buckets,
    *
FROM Sales.Orders


(10 rows affected)

Buckets | OrderID | ProductID | CustomerID | SalesPersonID | OrderDate  | ShipDate   | OrderStatus | ShipAddress        | BillAddress   | Quantity | Sales | CreationTime               
--------+---------+-----------+------------+---------------+------------+------------+-------------+--------------------+---------------+----------+-------+----------------------------
1       | 1       | 101       | 2          | 3             | 2025-01-01 | 2025-01-05 | Delivered   | 9833 Mt. Dias Blv. | 1226 Shoe St. | 1        | 10    | 2025-01-01 12:34:56.0000000
1       | 2       | 102       | 3          | 3             | 2025-01-05 | 2025-01-10 | Shipped     | 250 Race Court     | NULL          | 1        | 15    | 2025-01-05 23:22:04.0000000
1       | 3       | 101       | 1          | 5             | 2025-01-10 | 2025-01-25 | Delivered   | 8157 W. Book       | 8157 W. Book  | 2        | 20    | 2025-01-10 18:24:08.0000000
1       | 4       | 105       | 1          | 3         

In [4]:
-- CUME_DIST() Function : Cumulative Distribution calculates the distribution of data points within a window CUME_DIST = position number / Number of Rows 
-- Tie Rule : The position of the last occurrence of the same value
-- The CUME_DIST is a inclusive and (The current row is included)
-- Find the products that fail within the highest 40% of the prices
SELECT 
*,
CONCAT(DistRank * 100, '%') DistRankperc
FROM(
    SELECT
        Product,
        price,
        CUME_DIST() OVER (ORDER BY price DESC) DistRank
    from Sales.Products
)t WHERE DistRank <= 0.4

(2 rows affected)

Product | price | DistRank | DistRankperc
--------+-------+----------+-------------
Gloves  | 30    | 0.2      | 20%         
Caps    | 25    | 0.4      | 40%         
(2 rows)

Total execution time: 00:00:01.017

In [8]:
-- PERCENT_RANK() Function : Calculates the relative position of each row (Formula PERCENT_RANK = position number -1 / Number of Rows -1)
-- Tie Rule : The position of the first occurrence of the same value
-- The PERCENT_RANK is a exclusive and (The current row is excluded)
SELECT 
*,
CONCAT(percentrank * 100, '%') percentrankbyper
FROM(
    SELECT 
        product,
        price,
        PERCENT_RANK() OVER (ORDER BY price DESC) percentRank 
    FROM sales.products
)t WHERE percentrank <= 1

(5 rows affected)

product | price | percentRank | percentrankbyper
--------+-------+-------------+-----------------
Gloves  | 30    | 0           | 0%              
Caps    | 25    | 0.25        | 25%             
Socks   | 20    | 0.5         | 50%             
Tire    | 15    | 0.75        | 75%             
Bottle  | 10    | 1           | 100%            
(5 rows)

Total execution time: 00:00:01.019

In [ ]:
-- What are VAlue window functions: The VALUE window functions are used to access a value from other row
